# Explainer notebook: Danish housing affordability

Denne notebook samler koden bag projektet. Tekstafsnittene er kun gjort klar som korte placeholders, så de kan udfyldes med vores egne forklaringer senere.

## 1. Motivation

**Skriv selv:** Hvad handler projektet om, og hvorfor er det relevant?

- Hvilket spørgsmål prøver vi at svare på?
- Hvad skal læseren tage med fra websitet?

## 2. Dataset

**Skriv selv:** Beskriv kort de datasæt der bruges.

- Indkomstdata
- Kvadratmeterpriser
- Befolkning/flytning
- Kommune GeoJSON

Forklar også hvorfor datasættene passer til historien.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import plotly.express as px

# Basic settings
DATA_DIR = Path("data")
CLEANED_DIR = DATA_DIR / "cleaned"
BASE_YEAR = 1992
PLOT_YEAR = 2024
TOP_N = 5

KOMMUNE_NORMALIZATION = {
    "Århus": "Aarhus",
    "Vesthimmerland": "Vesthimmerlands",
}

COLORS = {
    "red": "#9B2D20",
    "blue": "#1F6F78",
    "grey": "#E0E0E0",
    "grid": "#D8D8D8",
    "text": "#333333",
    "muted_text": "#666666",
}

In [ ]:
def normalize_municipality_names(df, column = "kommune"):
    df = df.copy()
    df[column] = df[column].replace(KOMMUNE_NORMALIZATION)
    return df


def load_data(cleaned_dir = CLEANED_DIR):
    income = pd.read_csv(cleaned_dir / "income.csv")
    house_prices = pd.read_csv(cleaned_dir / "ejendomsdata.csv")
    population = pd.read_csv(cleaned_dir / "population_clean.csv")

    income = normalize_municipality_names(income)
    house_prices = normalize_municipality_names(house_prices)
    population = normalize_municipality_names(population)

    if "år" in population.columns:
        population = population.rename(columns={"år": "year"})

    return income, house_prices, population


income_df, house_price_df, pop_df = load_data()

## 3. Basic stats og preprocessing

**Skriv selv:** Forklar de vigtigste valg i datarensning og preprocessing.

Punkter der kan nævnes:

- Kommunenavne rettes så datasættene kan merges.
- Kvartalsvise boligpriser samles til årlige gennemsnit.
- Vi laver et simpelt mål for boligtilgængelighed: hvor mange m² en gennemsnitsindkomst kan købe.
- Kommuner med manglende data i start- eller slutåret fjernes.

In [ ]:
def dataset_overview(df, name):
    return pd.DataFrame({
        "dataset": [name],
        "rows": [len(df)],
        "columns": [df.shape[1]],
        "missing_values": [int(df.isna().sum().sum())],
    })


overview = pd.concat([
    dataset_overview(income_df, "Income"),
    dataset_overview(house_price_df, "Housing prices"),
    dataset_overview(pop_df, "Population"),
], ignore_index=True)

overview

In [ ]:
def prepare_house_prices(house_prices):
    house_long = house_prices.melt(
        id_vars="kommune",
        var_name="quarter",
        value_name="price",
    )
    house_long["year"] = house_long["quarter"].str[:4].astype(int)

    return (
        house_long
        .groupby(["kommune", "year"], as_index=False)["price"]
        .mean()
        .rename(columns={"price": "avg_price"})
    )


def prepare_income(income):
    return (
        income
        .rename(columns={"år": "year"})
        [["kommune", "year", "income"]]
        .rename(columns={"income": "avg_income"})
    )


def prepare_municipality_panel(
    house_prices,
    income,
):
    house_yearly = prepare_house_prices(house_prices)
    income_yearly = prepare_income(income)

    return house_yearly.merge(
        income_yearly,
        on=["kommune", "year"],
        how="inner",
    )


municipality_df = prepare_municipality_panel(house_price_df, income_df)
municipality_df.head()

In [ ]:
def compute_net_migration(population):
    population_sorted = population.sort_values(["kommune", "year"])

    migration = (
        population_sorted
        .groupby("kommune")
        .agg(
            population_start=("population", "first"),
            population_end=("population", "last"),
        )
        .reset_index()
    )
    migration["net_flytning"] = migration["population_end"] - migration["population_start"]
    migration["net_flytning_pct"] = (
        migration["net_flytning"] / migration["population_start"] * 100
    )

    return migration


def add_affordability_metrics(
    municipality_panel,
    base_year = BASE_YEAR,
):
    df = municipality_panel.rename(columns={
        "avg_price": "price",
        "avg_income": "income",
    }).copy()

    # Boligtilgængelighed: hvor mange m² fire års gennemsnitsindkomst kan købe.
    df["sqm_affordable"] = (4 * df["income"]) / df["price"]

    base = (
        df[df["year"] == base_year]
        [["kommune", "price", "income", "sqm_affordable"]]
        .rename(columns={
            "price": "price_base",
            "income": "income_base",
            "sqm_affordable": "sqm_affordable_base",
        })
    )

    df = df.merge(base, on="kommune", how="inner")

    df["price_change_pct"] = (df["price"] / df["price_base"] - 1) * 100
    df["income_change_pct"] = (df["income"] / df["income_base"] - 1) * 100
    df["sqm_change_abs"] = df["sqm_affordable"] - df["sqm_affordable_base"]
    df["sqm_change_pct"] = (df["sqm_affordable"] / df["sqm_affordable_base"] - 1) * 100
    df["sqm_index"] = df["sqm_affordable"] / df["sqm_affordable_base"] * 100

    return df


def filter_complete_start_end(
    df,
    years = (BASE_YEAR, PLOT_YEAR),
    required_columns = ("price", "income"),
):
    valid = []
    removed = []

    for kommune, group in df.groupby("kommune"):
        is_valid = True
        for year in years:
            year_rows = group[group["year"] == year]
            if year_rows.empty or year_rows[list(required_columns)].isna().any(axis=None):
                is_valid = False
                break

        if is_valid:
            valid.append(kommune)
        else:
            removed.append(kommune)

    return df[df["kommune"].isin(valid)].copy(), sorted(removed)


net_migration_df = compute_net_migration(pop_df)
municipality_affordability = add_affordability_metrics(municipality_df)
municipality_affordability_clean, removed_municipalities = filter_complete_start_end(
    municipality_affordability
)

print("Municipalities removed due to missing start/end data:", removed_municipalities)
municipality_affordability_clean.head()

In [ ]:
def prepare_national_trends(
    municipality_affordability,
    base_year = BASE_YEAR,
):
    national = (
        municipality_affordability
        .groupby("year", as_index=False)
        .agg(
            avg_price=("price", "mean"),
            avg_income=("income", "mean"),
        )
    )

    base = national.loc[national["year"] == base_year].iloc[0]
    national["price_change_pct"] = (national["avg_price"] / base["avg_price"] - 1) * 100
    national["income_change_pct"] = (national["avg_income"] / base["avg_income"] - 1) * 100

    return national


def get_plot_year_data(
    df,
    plot_year = PLOT_YEAR,
    exclude_municipalities = (),
):
    plot_df = df[df["year"] == plot_year].copy()
    if exclude_municipalities:
        plot_df = plot_df[~plot_df["kommune"].isin(exclude_municipalities)]
    return plot_df


national_trends = prepare_national_trends(municipality_affordability_clean)
plot_year_df = get_plot_year_data(municipality_affordability_clean)
plot_year_df.head()

### Første EDA-plot: national udvikling i priser og indkomst

**Skriv selv:** Hvad er hovedpointen i figuren?

In [ ]:
def plot_price_income_trends(plot_df):
    fig, ax = plt.subplots(figsize=(11, 6.3))

    ax.plot(
        plot_df["year"],
        plot_df["price_change_pct"],
        color=COLORS["red"],
        linewidth=3,
        label="Average square meter price",
        solid_capstyle="round",
        zorder=3,
    )
    ax.plot(
        plot_df["year"],
        plot_df["income_change_pct"],
        color=COLORS["blue"],
        linewidth=3,
        label="Average income before tax",
        solid_capstyle="round",
        zorder=3,
    )

    latest_year = plot_df["year"].max()
    latest = plot_df.loc[plot_df["year"] == latest_year].iloc[0]
    price_end = latest["price_change_pct"]
    income_end = latest["income_change_pct"]

    ax.hlines(
        y=price_end,
        xmin=plot_df["year"].min(),
        xmax=latest_year,
        color=COLORS["red"],
        linewidth=1.2,
        linestyle=(0, (2, 4)),
        alpha=0.55,
        zorder=1,
    )
    ax.hlines(
        y=income_end,
        xmin=plot_df["year"].min(),
        xmax=latest_year,
        color=COLORS["blue"],
        linewidth=1.2,
        linestyle=(0, (2, 4)),
        alpha=0.55,
        zorder=1,
    )

    label_x = latest_year + 0.3
    ax.text(label_x, price_end + 13, f"+{price_end:.0f}%", color=COLORS["red"], fontsize=11, fontweight="bold", va="center")
    ax.text(label_x, income_end + 13, f"+{income_end:.0f}%", color=COLORS["blue"], fontsize=11, fontweight="bold", va="center")
    ax.text(label_x, price_end - 12, "Square-Meter Price", color=COLORS["red"], fontsize=12, fontweight="bold", va="center")
    ax.text(label_x, income_end - 12, "Income", color=COLORS["blue"], fontsize=12, fontweight="bold", va="center")

    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel(f"percentage increase since {BASE_YEAR}", fontsize=11)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))

    ax.set_xlim(plot_df["year"].min(), 2025)
    ax.set_xticks([1992, 1995, 2000, 2005, 2010, 2015, 2020, 2024])

    ymax = max(plot_df["price_change_pct"].max(), plot_df["income_change_pct"].max())
    ax.set_ylim(-25, ymax * 1.12)

    ax.grid(axis="y", color=COLORS["grid"], linewidth=0.8, alpha=0.8)
    ax.grid(axis="x", visible=False)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#BBBBBB")
    ax.tick_params(axis="both", labelsize=12, colors=COLORS["text"])
    ax.legend().remove()

    plt.tight_layout(rect=[0, 0.05, 1, 0.92])
    return fig, ax


fig_price_income, ax_price_income = plot_price_income_trends(national_trends)

# Export disabled to avoid overwriting files used in the article.
# fig_price_income.savefig("output/price_income_trends.png", dpi=300, bbox_inches="tight")

plt.show()

## 4. Data analysis

**Skriv selv:** Forklar hvad analysen viser.

Dette er afsnittet med de vigtigste beregninger bag historien: boligtilgængelighed, kommunerangeringer, flytning og eventuelle ekstra analyser.

In [ ]:
def top_bottom_by_metric(
    df,
    metric,
    n = TOP_N,
):
    return pd.concat([
        df.nlargest(n, metric),
        df.nsmallest(n, metric),
    ]).sort_values(metric)


def clean_municipality_labels(series: pd.Series):
    return series.astype(str).str.replace(" -", "", regex=False).str.strip()

### Prisudvikling på kommuneniveau

**Skriv selv:** Forklar hvorfor top- og bundkommunerne er relevante for historien.

In [ ]:
def plot_price_change_top_bottom(
    df,
    plot_year = PLOT_YEAR,
    top_n = TOP_N,
):
    plot_df = get_plot_year_data(df, plot_year).dropna(subset=["price_change_pct"])
    top_bottom = top_bottom_by_metric(plot_df, "price_change_pct", top_n)

    top_municipalities = set(plot_df.nlargest(top_n, "price_change_pct")["kommune"])
    colors = [
        COLORS["blue"] if kommune in top_municipalities else COLORS["red"]
        for kommune in top_bottom["kommune"]
    ]

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top_bottom["kommune"], top_bottom["price_change_pct"], color=colors, alpha=0.9)

    xmax = top_bottom["price_change_pct"].max()
    tick_max = int(np.ceil(xmax / 100) * 100)
    ax.set_xticks(np.arange(100, tick_max + 100, 200))
    ax.set_xlim(0, tick_max * 1.05)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))

    ax.set_xlabel("Price Change in percentage", fontsize=12)
    ax.set_ylabel("")
    ax.grid(axis="x", color=COLORS["grid"], linewidth=0.8, alpha=0.8)
    ax.grid(axis="y", visible=False)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#BBBBBB")
    ax.tick_params(axis="both", labelsize=16, colors=COLORS["text"])

    plt.tight_layout()
    return fig, ax


fig_price_change, ax_price_change = plot_price_change_top_bottom(municipality_affordability_clean)

# Export disabled to avoid overwriting files used in the article.
# fig_price_change.savefig("output/price_change_top_bottom.png", dpi=300)

plt.show()

### Ændring i antal m² man har råd til

**Skriv selv:** Forklar målet, og hvorfor det er nemmere at forstå end prisudvikling alene.

In [ ]:
def plot_sqm_change_top_bottom(
    df,
    plot_year = PLOT_YEAR,
    top_n = TOP_N,
):
    plot_df = get_plot_year_data(df, plot_year).dropna(subset=["sqm_change_abs"])
    plot_afford = top_bottom_by_metric(plot_df, "sqm_change_abs", top_n).copy()
    plot_afford["kommune"] = clean_municipality_labels(plot_afford["kommune"])

    colors = np.where(plot_afford["sqm_change_abs"] < 0, COLORS["red"], COLORS["blue"])

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(plot_afford["kommune"], plot_afford["sqm_change_abs"], color=colors, alpha=0.95)
    ax.axvline(0, color="#555555", linewidth=1.2)

    xmin = plot_afford["sqm_change_abs"].min()
    xmax = plot_afford["sqm_change_abs"].max()
    left_pad = max(5, abs(xmin) * 0.18)
    right_pad = max(5, abs(xmax) * 0.12)
    ax.set_xlim(xmin - left_pad, xmax + right_pad)

    for i, value in enumerate(plot_afford["sqm_change_abs"]):
        if value >= 0:
            ax.text(
                value + right_pad * 0.15,
                i,
                f"+{value:.0f} m²",
                va="center",
                ha="left",
                fontsize=12,
                color=COLORS["text"],
            )
        else:
            ax.text(
                value - left_pad * 0.15,
                i,
                f"{value:.0f} m²",
                va="center",
                ha="right",
                fontsize=12,
                color=COLORS["text"],
            )

    ax.set_xlabel(f"difference in affordable square meters {BASE_YEAR}-{PLOT_YEAR}", fontsize=14)
    ax.set_ylabel("")
    ax.grid(axis="x", color=COLORS["grid"], linewidth=0.8, alpha=0.8)
    ax.grid(axis="y", visible=False)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#BBBBBB")
    ax.tick_params(axis="both", labelsize=16, colors=COLORS["text"])

    plt.tight_layout()
    return fig, ax


fig_sqm_change, ax_sqm_change = plot_sqm_change_top_bottom(municipality_affordability_clean)

# Export disabled to avoid overwriting files used in the article.
# fig_sqm_change.savefig("output/sqm_change_top_bottom.png", dpi=300)

plt.show()

## 5. Genre og narrativt design

**Skriv selv:** Forklar hvilken type data story websitet bruger, og forbind det til Segel og Heer.

Mulige ting at nævne:

- Genre: fx magazine style, annotated chart, partitioned poster, martini glass eller drill-down story.
- Visual Narrative: visual structuring, highlighting og transition guidance.
- Narrative Structure: ordering, interactivity og messaging.

## 6. Visualizations

**Skriv selv:** Forklar de visualiseringer der bruges, og hvorfor de passer til historien.

### Interaktive kort

**Skriv selv:** Forklar hvorfor kortene er gode til geografisk udforskning, og hvordan læseren skal bruge dem.

In [ ]:
def load_geojson(path = DATA_DIR / "kommuneinddeling.geojson"):
    with path.open("r", encoding="utf-8") as f:
        geojson = json.load(f)

    geojson["features"] = [
        feature for feature in geojson["features"]
        if feature["properties"]["navn"] not in ["Bornholm", "Ærø"]
    ]
    return geojson


def prepare_map_dataframe(
    affordability_df,
    migration_df,
    plot_year = PLOT_YEAR,
):
    base_df = get_plot_year_data(
        affordability_df,
        plot_year=plot_year,
        exclude_municipalities=("Bornholm",),
    )

    base_df = base_df.merge(
        migration_df[["kommune", "net_flytning_pct"]],
        on="kommune",
        how="left",
    )

    q10 = base_df["price_change_pct"].quantile(0.15)
    q90 = base_df["price_change_pct"].quantile(0.85)
    base_df["price_color"] = base_df["price_change_pct"].clip(lower=q10, upper=q90)

    return base_df


geojson = load_geojson()
map_base_df = prepare_map_dataframe(municipality_affordability_clean, net_migration_df)
map_base_df.head()

In [ ]:
MAPBOX_KWARGS = dict(
    geojson=geojson,
    locations="kommune",
    featureidkey="properties.navn",
    color_continuous_scale=[COLORS["red"], COLORS["grey"], COLORS["blue"]],
    mapbox_style="carto-positron",
    center={"lat": 56.2, "lon": 10.5},
    zoom=6.4,
    height=900,
    opacity=0.8,
)

STATIC_CONFIG = {
    "scrollZoom": False,
    "doubleClick": False,
    "displayModeBar": False,
}


def create_map(
    df,
    color_col,
    title,
    customdata_cols,
    hover_template,
    midpoint = None,
):
    kwargs = MAPBOX_KWARGS.copy()
    if midpoint is not None:
        kwargs["color_continuous_midpoint"] = midpoint

    fig = px.choropleth_mapbox(
        df,
        color=color_col,
        title=title,
        **kwargs,
    )

    fig.update_traces(
        customdata=df[customdata_cols].round(1).values,
        hovertemplate=hover_template,
    )
    fig.update_layout(
        coloraxis_showscale=False,
        dragmode=False,
        hovermode="closest",
        title_font_color="black",
    )

    return fig


def add_manual_legend(fig, legend_items, y_start = 0.98):
    for i, item in enumerate(legend_items):
        fig.add_annotation(
            x=0.01,
            y=y_start - i * 0.03,
            xref="paper",
            yref="paper",
            text=f"<span style='color:{item['color']};'>■</span> {item['label']}",
            showarrow=False,
            font=dict(size=16),
            align="left",
        )
    return fig


def build_interactive_maps(df):
    median_price_change = df["price_change_pct"].median()

    fig_aff = create_map(
        df=df,
        color_col="sqm_change_abs",
        title=f"<b>Change in affordable square meters since {BASE_YEAR} to {PLOT_YEAR}</b>",
        customdata_cols=["sqm_affordable", "sqm_change_abs"],
        hover_template=(
            "<b>%{location}</b><br>"
            "Sqm affordable: %{customdata[0]:.0f} m²<br>"
            f"Change since {BASE_YEAR}: "
            "%{customdata[1]:+.0f} m²<extra></extra>"
        ),
    )
    add_manual_legend(fig_aff, [
        {"label": f"Less affordable than {BASE_YEAR}", "color": COLORS["red"]},
        {"label": "No change", "color": COLORS["grey"]},
        {"label": f"More affordable than {BASE_YEAR}", "color": COLORS["blue"]},
    ])

    fig_move = create_map(
        df=df,
        color_col="net_flytning_pct",
        title=f"<b>Net migration {BASE_YEAR} to {PLOT_YEAR}</b>",
        customdata_cols=["net_flytning_pct", "sqm_change_abs"],
        hover_template=(
            "<b>%{location}</b><br>"
            "Net migration: %{customdata[0]:.1f}%<br>"
            "Change in sqm affordable: %{customdata[1]:+.0f} m²<extra></extra>"
        ),
    )
    add_manual_legend(fig_move, [
        {"label": "Net out-migration", "color": COLORS["red"]},
        {"label": "Little/no change", "color": COLORS["grey"]},
        {"label": "Net in-migration", "color": COLORS["blue"]},
    ])

    fig_price = create_map(
        df=df,
        color_col="price_color",
        title=f"<b>Relative square-meter price growth from {BASE_YEAR} to {PLOT_YEAR}</b>",
        customdata_cols=["price_change_pct"],
        hover_template="<b>%{location}</b><br>Price change: %{customdata[0]:+.1f}%<extra></extra>",
        midpoint=median_price_change,
    )
    add_manual_legend(fig_price, [
        {"label": "Lower price growth", "color": COLORS["red"]},
        {"label": "Middle of distribution", "color": COLORS["grey"]},
        {"label": "Higher price growth", "color": COLORS["blue"]},
    ])

    return {
        "affordability": fig_aff,
        "migration": fig_move,
        "price_change": fig_price,
    }


interactive_maps = build_interactive_maps(map_base_df)

# Gemme-linjerne er slået fra, så artikel-filerne ikke overskrives.
# interactive_maps["affordability"].write_html("output/affordability.html", full_html=True, include_plotlyjs="cdn", config=STATIC_CONFIG)
# interactive_maps["migration"].write_html("output/migration.html", full_html=True, include_plotlyjs="cdn", config=STATIC_CONFIG)
# interactive_maps["price_change"].write_html("output/price_change.html", full_html=True, include_plotlyjs="cdn", config=STATIC_CONFIG)

interactive_maps["affordability"].show(config=STATIC_CONFIG)
interactive_maps["migration"].show(config=STATIC_CONFIG)
interactive_maps["price_change"].show(config=STATIC_CONFIG)

### Data til dashboard

Denne celle laver den JSON-struktur som websitet bruger. Selve eksporten er kommenteret ud, så filen fra artiklen ikke bliver overskrevet.

In [ ]:
def safe_index(value, base):
    if pd.isna(value) or pd.isna(base) or base == 0:
        return None
    return round((value / base) * 100, 1)


def safe_int(value):
    if pd.isna(value):
        return None
    return int(round(value))


def prepare_yearly_population(population):
    pop_yearly = population[["kommune", "year", "population"]].copy()
    pop_yearly = pop_yearly.sort_values(["kommune", "year"])
    pop_yearly["pop_change"] = pop_yearly.groupby("kommune")["population"].diff()
    pop_yearly["net_migration_pct_yearly"] = (
        pop_yearly["pop_change"] / pop_yearly.groupby("kommune")["population"].shift(1)
    ) * 100
    return pop_yearly


def filter_export_municipalities(
    df,
    start_year = BASE_YEAR,
    end_year = PLOT_YEAR,
):
    valid = []
    removed = []

    for kommune, group in df.groupby("kommune"):
        row_start = group[group["year"] == start_year]
        row_end = group[group["year"] == end_year]

        if row_start.empty or row_end.empty:
            removed.append(kommune)
            continue

        required_values = [
            row_start["price"].values[0],
            row_start["income"].values[0],
            row_end["price"].values[0],
            row_end["income"].values[0],
        ]

        if any(pd.isna(value) for value in required_values):
            removed.append(kommune)
        else:
            valid.append(kommune)

    return df[df["kommune"].isin(valid)].copy(), sorted(removed)


def build_dashboard_export(
    municipality_panel,
    population,
    start_year = BASE_YEAR,
    end_year = PLOT_YEAR,
):
    df_export = municipality_panel.rename(columns={
        "avg_price": "price",
        "avg_income": "income",
    }).copy()

    pop_yearly = prepare_yearly_population(population)

    df_export = df_export.merge(
        pop_yearly[["kommune", "year", "net_migration_pct_yearly"]],
        on=["kommune", "year"],
        how="left",
    ).sort_values(["kommune", "year"])

    df_export, removed = filter_export_municipalities(df_export, start_year, end_year)

    export_data = {}

    for kommune, group in df_export.groupby("kommune"):
        group = group.sort_values("year")
        row_start = group[group["year"] == start_year]
        row_end = group[group["year"] == end_year]

        base_price = row_start["price"].values[0]
        base_income = row_start["income"].values[0]
        price_end = row_end["price"].values[0]
        income_end = row_end["income"].values[0]

        yearly_migration = group["net_migration_pct_yearly"].fillna(0)
        cumulative_migration = yearly_migration.cumsum()

        pop_start_row = pop_yearly[(pop_yearly["kommune"] == kommune) & (pop_yearly["year"] == start_year)]
        pop_end_row = pop_yearly[(pop_yearly["kommune"] == kommune) & (pop_yearly["year"] == end_year)]

        pop_start = pop_start_row["population"].values[0] if not pop_start_row.empty else np.nan
        pop_end = pop_end_row["population"].values[0] if not pop_end_row.empty else np.nan
        net_migration_raw = None
        if pd.notna(pop_start) and pd.notna(pop_end):
            net_migration_raw = int(round(pop_end - pop_start))

        export_data[kommune] = {
            "years": group["year"].tolist(),
            "price_idx": [safe_index(price, base_price) for price in group["price"]],
            "income_idx": [safe_index(income, base_income) for income in group["income"]],
            "migration_idx": [round(100 + value, 1) for value in cumulative_migration],
            "price_pct": round((price_end / base_price - 1) * 100),
            "income_pct": round((income_end / base_income - 1) * 100),
            "migration_pct": round(cumulative_migration.iloc[-1], 1) if len(cumulative_migration) > 0 else 0,
            "price_1992": safe_int(base_price),
            "price_2024": safe_int(price_end),
            "income_1992": safe_int(base_income),
            "income_2024": safe_int(income_end),
            "population_1992": safe_int(pop_start),
            "population_2024": safe_int(pop_end),
            "net_migration_raw": net_migration_raw,
        }

    return export_data, removed


dashboard_export, removed_from_export = build_dashboard_export(municipality_df, pop_df)
print(f"Prepared dashboard data for {len(dashboard_export)} municipalities.")
print("Municipalities removed from dashboard export:", removed_from_export)

# Gemme-linjen er slået fra, så JSON-filen fra artiklen ikke overskrives.
# with open("output/municipal_data.json", "w", encoding="utf-8") as f:
#     json.dump(dashboard_export, f, ensure_ascii=False)

## 7. Discussion

**Skriv selv:** Tænk kritisk over resultatet.

- Hvad fungerede godt?
- Hvad mangler stadig?
- Hvad kunne forbedres, og hvorfor?
- Hvilke begrænsninger har data eller metode?

## 8. Contributions

**Skriv selv:** Skriv kort hvem der især havde ansvar for hvilke dele.

Undgå at skrive at alle bidrog lige meget. Beskriv i stedet fx hvem der tog lead på data cleaning, analyse, kort, website, tekst, design osv.

## 9. References

**Skriv selv:** Tilføj referencer med akademisk standard.

Eksempler:

- Segel, E. & Heer, J. (2010). *Narrative Visualization: Telling Stories with Data*.
- De datakilder der bruges i projektet.